# **Language Similarity Analysis**

This notebook identifies which languages are most linguistically similar to a target language — in this case **Filipino (`fil`)**.

Similarity is measured across four independent signals: phoneme inventory overlap (from Phoible), genetic relatedness, geographic proximity (both from URIEL via lang2vec), and shared writing script. These are combined into a single averaged distance score following the approach of Deri & Knight (2016).

The goal is to find the best candidate languages for cross-lingual transfer in a G2P (grapheme-to-phoneme) model — languages whose training data is most likely to be useful for Filipino.

In [1]:
import csv
import os
import urllib.request
import math
import lang2vec.lang2vec as l2v
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

from lang_distance import (
    phoneme_inventory_distance, script_row, normalize,
    lang2lang_average, closest_languages
)
from uriel_distance import uriel_row, covered_languages
from script_distance import script_row_auto

e:\College\Year 3\Term 3\DEEPLRN\Hunger\.venv\Lib\site-packages\lang2vec\lang2vec.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### **Phoible Data**

[Phoible](https://phoible.org/) is a cross-linguistic database of phonological inventories — the sets of consonants, vowels, and tones that each language uses. Each language entry is a binary vector over a shared feature space.

We load the largest available inventory per language (by segment count) and use it to compute phoneme inventory distance: two languages are close if they use a similar set of phonemes. This is one of the most direct signals of phonological similarity and, importantly, is based on real attested data for `fil`.

In [2]:
_PHOIBLE_FEATURES = {
    "InventoryID", "Glottocode", "ISO6393", "LanguageName", "SpecificDialect",
    "GlyphID", "Phoneme", "Allophones", "Marginal", "SegmentClass", "Source",
}

phoible_path = Path("../data/phoible.csv")

reader = csv.DictReader(open(phoible_path, encoding="utf-8"))
features = [c for c in reader.fieldnames if c not in _PHOIBLE_FEATURES]

by_lang = {}
id2name = {}
for row in reader:
    # Language ID
    lid = row["ISO6393"]
    id2name.setdefault(lid, row["LanguageName"].lower())

    if not lid or lid == "NA":
        continue

    vec = [
        (row[f].split(",")[0] if row else 0)
        for f in features
    ]

    by_lang.setdefault(lid, {}).setdefault(row["InventoryID"], []).append(vec)

phoible = {}

for lid, inventories in by_lang.items():
    phoible[lid] = max(inventories.values(), key=len)

## **Lang2Lang Distance**

We compute pairwise distances between the target language and a pool of candidates across four metrics, then average the min-max normalised scores:

| Column | Source | What it measures |
|--------|--------|-----------------|
| `inv` | Phoible | Cosine distance over phoneme inventory vectors |
| `gen` | URIEL `fam` | Cosine distance over language-family membership vectors |
| `geo` | URIEL `geo` | Cosine distance over geographic anchor coordinates |
| `scr` | Unicode | Whether languages share the same writing script |

### **Reliability note for `fil`**

URIEL's syntactic (`syn`) and phonological typology (`phon`) features are kNN-imputed for `fil` — it has no raw data in WALS, SSWL, or Ethnologue. The imputation uses genetic and geographic neighbors (Cebuano, Aklan, Bikol, etc.) as proxies, so distances to those exact languages come out as zero by construction. These two columns are excluded here because they would be circular rather than informative.

In [3]:
target = "fil" # maybe FIL also later?
k =  100
candidates = None
script_samples = None

METRIC_ORDER = ["phoneme", "genetic", "geographic", "script"]
SHORT = {"phoneme": "inv", "genetic": "gen", "geographic": "geo", "script": "scr"}

In [4]:
URIEL_METRICS = ("genetic", "geographic")

if target not in phoible:
    raise ValueError(f"{target!r} has no Phoible inventory")

pool = set(phoible) & covered_languages()
if candidates is not None:
    pool &= set(candidates)

pool.discard(target)

target_inv = phoible[target]
cand_inv = {c: phoible[c] for c in pool if c in phoible}

# URIEL

rows = {"phoneme": phoneme_inventory_distance(target_inv, cand_inv)}
for m in URIEL_METRICS:
    rows[m] = uriel_row(m, target, pool)

# SCRIPTS

# if script_samples and target in script_samples:
#     others = {c: script_samples[c] for c in pool if c in script_samples}
#     rows["scripts"] = script_row(script_samples[target], others)

rows["script"] = script_row_auto(target, pool)

base = set(rows["phoneme"])
for m in URIEL_METRICS:
    base &= set(rows[m])

norm = {}
for m, row in rows.items():
    sub = {c: row[c] for c in base if c in row}
    if sub:
        norm[m] = normalize(sub)

scores = lang2lang_average(norm.values())
close_langs = closest_languages(scores, k, base)
ranked = closest_languages(scores, k=k, candidates=base)
out = []
for lid, avg in ranked:
    per = {m: norm[m][lid] for m in norm if lid in norm[m]}
    out.append((lid, avg, per))


In [5]:
cols = [m for m in METRIC_ORDER]
header = f"{'lang':<6}" + "".join(f"{SHORT[m]:>7}" for m in cols) + f"{'AVG':>8}"
print(f"closest languages to {target!r}  (lower = more similar)")
print(header)
print("-" * len(header))
for lid, avg, per in out:
    cells = "".join(
        (f"{per[m]:>7.2f}" if m in per else f"{'-':>7}") for m in cols
    )
    print(f"{lid:<6}{cells}{avg:>8.3f}")

closest languages to 'fil'  (lower = more similar)
lang      inv    gen    geo    scr     AVG
------------------------------------------
tgl      0.00   0.00   0.00   0.00   0.000
ceb      0.00   0.23   0.00   0.00   0.057
bcl      0.00   0.29   0.00   0.00   0.071
akl      0.00   0.33   0.00   0.00   0.083
tsg      0.00   0.33   0.01   0.00   0.084
hil      0.00   0.37   0.00   0.00   0.093
cha      0.00   0.35   0.04   0.00   0.095
pau      0.04   0.35   0.01   0.00   0.098
blf      0.02   0.38   0.01   0.00   0.103
ilo      0.00   0.43   0.00   0.00   0.109
tiy      0.00   0.43   0.00   0.00   0.109
ivv      0.00   0.43   0.01   0.00   0.110
pam      0.02   0.43   0.00   0.00   0.113
gay      0.00   0.43   0.06   0.00   0.123
pwn      0.02   0.47   0.01   0.00   0.123
duo      0.00   0.49   0.00   0.00   0.124
mwt      0.02   0.43   0.05   0.00   0.124
dbj      0.00   0.49   0.01   0.00   0.125
mkm      0.02   0.43   0.05   0.00   0.125
mad      0.00   0.49   0.04   0.00   0.133
tes

### **Results**
Each row is a candidate language ranked by average distance to `fil` (lower = more similar). A score of `0.00` in a column means the candidate is maximally similar to `fil` on that metric after normalisation — not that the raw distance is zero.

The candidate pool is the intersection of Phoible coverage and URIEL coverage, so only languages with both a phoneme inventory and genetic/geographic data appear.

In [6]:
results = list()
for tup in close_langs:
    
    results.append((tup[0], id2name[tup[0]], tup[1]))
results

[('tgl', 'tagalog', np.float64(8.825418456427597e-05)),
 ('ceb', 'cebuano', np.float64(0.05740864781612022)),
 ('bcl', 'bikol (bicolano)', np.float64(0.07143984273473628)),
 ('akl', 'aklan', np.float64(0.08296172380473475)),
 ('tsg', 'tausug (suluk)', np.float64(0.08437570988876521)),
 ('hil', 'hiligaynon', np.float64(0.09255614690458612)),
 ('cha', 'chamorro', np.float64(0.09533437840062599)),
 ('pau', 'palauan', np.float64(0.09796102509224104)),
 ('blf', 'buol', np.float64(0.10262891514277521)),
 ('ilo', 'ilocano', np.float64(0.10862069860949188)),
 ('tiy', 'tiruray', np.float64(0.10884998944884507)),
 ('ivv', 'ivatan', np.float64(0.10961223289987036)),
 ('pam', 'kapampangan', np.float64(0.11286876115541249)),
 ('gay', 'gayo', np.float64(0.12251961446640464)),
 ('pwn', 'paiwan', np.float64(0.12294572726641786)),
 ('duo',
  'dupaningan agta; eastern cagayan agta; dupaninan agta',
  np.float64(0.12375638728576481)),
 ('mwt', 'moken', np.float64(0.12431512633902787)),
 ('dbj', "ida''an'

### **Interpretation**

The top results are overwhelmingly Philippine and broader Austronesian languages, which is expected. A few observations:

- **Tagalog (`tgl`)** ranks first with a near-zero distance — Filipino is the standardised register of Tagalog, so they share the same family membership vector, near-identical geographic coordinates, and the same Phoible inventory. It is the single most transferable language for a Filipino G2P model.
- **Cebuano (`ceb`), Bikol (`bcl`), Aklan (`akl`), Hiligaynon (`hil`), Ilocano (`ilo`), Kapampangan (`pam`)** are all Philippine languages with zero phoneme inventory distance, meaning they use the same set of phonemes as Filipino. Their separation in the ranking comes entirely from genetic distance (how closely related they are within the Austronesian tree).
- **Chamorro (`cha`) and Palauan (`pau`)** are Micronesian languages — geographically close and in the same macro-family, but genetically more distant, which pushes them down the ranking despite small phoneme inventory distances.
- Further down the list, Malay/Indonesian varieties (`zsm`, `ind`, `xmm`, etc.) and Sulawesi languages appear — same broad Malayo-Polynesian branch but more genetically distant and geographically further.